# Aula 5.2 - Alura - versao aprimorada\n

In [ ]:

import re
from datetime import datetime

dbutils.widgets.text("gravar", "false")
dbutils.widgets.text("catalogo", "workspace")
dbutils.widgets.text("esquema", "default")

gravar = dbutils.widgets.get("gravar").strip().lower() == "true"
cat = dbutils.widgets.get("catalogo").strip()
esq = dbutils.widgets.get("esquema").strip()

for identificador in [cat, esq]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z_0-9]*", identificador):
        raise ValueError("Nome de catalogo ou esquema invalido")

dbutils.widgets.text("data_inicio", "20260101")
dbutils.widgets.text("data_fim", "20260110")

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# 1. Parâmetros do Job

p_inicio = dbutils.widgets.get("data_inicio")
p_fim = dbutils.widgets.get("data_fim")

# p_inicio = "20260101"
# p_fim = "20260110"

print(f"Gerando dados de produção (Injeção de Drift) para o período: {p_inicio} a {p_fim}")

def gerar_producao_periodo(spark, dt_inicio_str, dt_fim_str, n_por_dia=100):
    
    # Gerando os dias particionados de forma aleatória dentro do range
    dt_inicio = datetime.strptime(dt_inicio_str, '%Y%m%d')
    dt_fim = datetime.strptime(dt_fim_str, '%Y%m%d')
    dias_delta = (dt_fim - dt_inicio).days
    if not 1 <= dias_delta <= 366:
        raise ValueError("Intervalo invalido: 1 a 366 dias")
    n_total = n_por_dia * dias_delta
    
    np.random.seed(int(dt_inicio_str))
    regioes = np.random.choice(
        ['nordeste', 'norte', 'sudeste', 'sul', 'outro'],
        n_total,
        p=[0.2, 0.1, 0.4, 0.2, 0.1]
    )

    random_offsets = np.repeat(np.arange(dias_delta), n_por_dia)
    np.random.shuffle(random_offsets)

    dias_particao = [(dt_inicio + timedelta(days=int(offset))).strftime("%Y%m%d") for offset in random_offsets]
    
    pdf_prod = pd.DataFrame({
        "dia_prtc": dias_particao, # Coluna de partição
        "renda_mensal_k": np.random.uniform(0, 2000, n_total),
        "tempo_medio_clique_segundos": np.random.uniform(10, 30, n_total),
        "media_interacoes_suporte": np.random.uniform(30, 200, n_total),
        "media_cupons_ativos": np.random.uniform(0, 15, n_total),
        "media_score_nps_cliente": np.random.randint(0, 8, n_total),
        "media_dias_inatividade": np.random.randint(0, 120, n_total),
        "total_gasto_acumulado_reais":np.random.uniform(1000, 20500, n_total),
        "genero_cliente_m": np.random.binomial(1, 0.48, n_total),
        "regiao_cliente_nordeste": (regioes == 'nordeste').astype(int),
        "regiao_cliente_norte": (regioes == 'norte').astype(int),
        "regiao_cliente_sudeste": (regioes == 'sudeste').astype(int),
        "regiao_cliente_sul": (regioes == 'sul').astype(int)
    })
    
    pdf_prod[
        ["media_score_nps_cliente", "media_dias_inatividade"]
    ] = pdf_prod[
        ["media_score_nps_cliente", "media_dias_inatividade"]
    ].astype(float)

    # Target Prod
    prob_prod = 1 / (1 + np.exp(-np.clip(pdf_prod['total_gasto_acumulado_reais']*0.1 + pdf_prod['media_interacoes_suporte']*0.05 - 300, -50, 50)))
    pdf_prod['comprou_eletronico'] = np.random.binomial(1, prob_prod)

    schema_dados = StructType([
        StructField("dia_prtc", StringType(), True),
        StructField("renda_mensal_k", DoubleType(), True),
        StructField("tempo_medio_clique_segundos", DoubleType(), True),
        StructField("media_interacoes_suporte", DoubleType(), True),
        StructField("media_cupons_ativos", DoubleType(), True),
        StructField("media_score_nps_cliente", DoubleType(), True),
        StructField("media_dias_inatividade", DoubleType(), True),
        StructField("total_gasto_acumulado_reais", DoubleType(), True),
        StructField("genero_cliente_m", IntegerType(), True),
        StructField("regiao_cliente_nordeste", IntegerType(), True),
        StructField("regiao_cliente_norte", IntegerType(), True),
        StructField("regiao_cliente_sudeste", IntegerType(), True),
        StructField("regiao_cliente_sul", IntegerType(), True),
        StructField("comprou_eletronico", IntegerType(), True)
    ])

    return spark.createDataFrame(pdf_prod, schema=schema_dados)

# 2. Geração e gravação particionada
df_prod = gerar_producao_periodo(spark, p_inicio, p_fim)
nome_tabela_prod = f"{cat}.{esq}.base_tabela_prod_aprimorada"

qtd = df_prod.count()

dias = (
    datetime.strptime(p_fim, "%Y%m%d")
    - datetime.strptime(p_inicio, "%Y%m%d")
).days

assert qtd == dias * 100

print("PRODUCAO:", qtd, "registros")

df_prod.groupBy("dia_prtc").count().orderBy(
    "dia_prtc"
).show(366)

if gravar:
    if spark.catalog.tableExists(nome_tabela_prod):
        df_prod.write.format("delta").mode("overwrite").option(
            "replaceWhere",
            f"dia_prtc >= '{p_inicio}' AND dia_prtc < '{p_fim}'"
        ).saveAsTable(nome_tabela_prod)
    else:
        df_prod.write.format("delta").mode("overwrite").partitionBy(
            "dia_prtc"
        ).saveAsTable(nome_tabela_prod)

    print("GRAVADO:", nome_tabela_prod)
else:
    print("PREVIA: producao nao gravada")